In [ ]:
# notebook to look at examples of emvars in tf seqlets within the same enhancer

In [1]:
# import packages
import pandas as pd
import os
import pybedtools
from tqdm import tqdm
from collections import Counter

In [17]:
def multiTF_analysis (path2distalCREpreds, 
                      path2distalCREseqlets_k562,
                      path2distalCREseqlets_hepg2,
                      path2distalCREseqlets_sknsh,
                      vierstra_dict):
    # define function for more quickly reading in predictions
    def read_in_sat_mut (path2satmut, chunksize=1000000):
        # define a list for concatenating #
        chunks2cat = []
        count = 0
        # iterate through chunked tsv #
        for chunk in tqdm(pd.read_csv(path2satmut, sep = '\t', chunksize=chunksize)):
            chunks2cat.append(chunk)
            count +=1
            if count > 10:
                break
        # concatenate chunks #
        cat_df = pd.concat(chunks2cat)
        return cat_df
    # define function to convert emVar DF to a BED file
    def df2bed (emvar_df):
        chrom4bed = []
        start4bed = []
        end4bed = []
        id4bed = []
        for chrom, pos, id, ref, alt in zip(emvar_df['chrom'], 
                                            emvar_df['pos'], 
                                            emvar_df['id'], 
                                            emvar_df['ref'], 
                                            emvar_df['alt']):
            # update chromosome and id lists
            chrom4bed.append(chrom)
            id4bed.append(f'{(':').join([chrom, str(pos), ref, alt])}_{id}')
            # check the length of the variant for generating the start and stop intervals
            if len(ref) == 1 and len(alt) == 1: # SNPs
                start4bed.append(pos - 1)
                end4bed.append(pos)
            elif len(ref) < len(alt): # Insertions
                start4bed.append(pos)
                end4bed.append(pos)
            elif len(ref) > len(alt): # Deletions
                start4bed.append(pos)
                end4bed.append(pos + len(ref) - 1)
        bed2return = pybedtools.BedTool.from_dataframe(pd.DataFrame({0 : chrom4bed,
                                                                     1 : start4bed,
                                                                     2 : end4bed,
                                                                     3 : id4bed}).drop_duplicates()).sort()
        return bed2return
    # define updated function for more flexible tf assignment
    def create_long_form_tf_df(raw_bed_df, vierstra_dict):
        """
        Takes the raw BED file and "explodes" the 4th column, creating a 
        new long-form DataFrame with one row per TF hit.

        Args:
            raw_bed_df (pd.DataFrame): The raw BED file from bedmap.

        Returns:
            pd.DataFrame: A long-form DataFrame with parsed TF info.
        """
        # Make sure the 4th column (index 3) is a string
        raw_bed_df[3] = raw_bed_df['name'].astype(str)
        
        # Split the string by ';' into a list
        df_split = raw_bed_df.assign(tf_hits_list = raw_bed_df[3].str.split(';'))
        
        # Explode the list into new rows.
        # The original columns (0, 1, 2) will be duplicated.
        long_df = df_split.explode('tf_hits_list').reset_index(drop=True)
        
        # --- Now, parse the single hit string on each row ---
        
        # Helper function to safely parse scores
        def parse_score(hit_string):
            try:
                return float(hit_string.split('_')[-1])
            except (ValueError, IndexError):
                return 0.0

        # Parse the enhancer, TF, and contribution score from the string
        long_df['full_hit_string'] = long_df['tf_hits_list']
        long_df['hocomoco_tf'] = long_df['full_hit_string'].apply(lambda x: x.split('_EH')[0])
        long_df['representative_tf'] = long_df['hocomoco_tf'].apply(lambda x: x.split('_')[0])
        long_df['enhancer_id'] = long_df['full_hit_string'].apply(lambda x: x.split('_')[2])
        long_df['rep_tf_contrib'] = long_df['full_hit_string'].apply(parse_score)
        long_df.loc[:,'vierstra_cluster'] = [vierstra_dict.get(i) for i in long_df['hocomoco_tf']]
        # Add activity class for consistency
        long_df['activity_class'] = long_df['rep_tf_contrib'].apply(
            lambda s: 'Activator' if s > 0 else ('Repressor' if s < 0 else 'Neutral')
        )
        
        # Clean up intermediate columns
        final_df = long_df.drop(columns=['tf_hits_list', 3])
        
        return final_df
    # define function for getting multi-tf enhancers
    def get_multiTF_enhancers_from_long_df_vClustered (long_df):
        """
        Finds enhancers with multiple instances of a single TF 
        from the long-form DataFrame.
        """
        # 1. Find groups with size > 1
        #    In our example, (EH35, GATA) has size 2. (EH35, SNAI1) has size 1.
        grouped = long_df.groupby(['thickEnd', 'vierstra_cluster']).transform('size')
        
        # 2. Get the *rows* that are part of these multi-instance groups
        #    This will select both GATA rows.
        multiTFs = long_df[grouped > 1]
        
        # 3. Get the *enhancer IDs* that contain these multi-instance TFs
        #    This will get 'EH35'.
        multiTF_enhancer_ids = multiTFs['thickEnd'].unique()
        
        # 4. Filter the original long_df for *all* rows belonging to those enhancers
        #    This returns all 3 rows (both GATAs and the SNAI1) for enhancer EH35.
        final_df = long_df[long_df['thickEnd'].isin(multiTF_enhancer_ids)]
        
        # 5. The 'rep_tf_contrib' column is *already on the DataFrame*.
        #    No complex parsing is needed.
        
        return final_df.sort_values(['thickEnd', 'start']) # '1' is the start col
    ### first ###
    # open all saturation mutagenesis predictions for that chromosome
    allPreds = read_in_sat_mut(path2distalCREpreds)
    # filter for emVars in each cell type
    # k562
    k_emvars = allPreds[allPreds['k562_skew_pred'].abs() > 0.5].copy()
    # hepg2
    h_emvars = allPreds[allPreds['hepg2_skew_pred'].abs() > 0.5].copy()
    # sknsh
    s_emvars = allPreds[allPreds['sknsh_skew_pred'].abs() > 0.5].copy()
    ### second ###
    # convert those DFs to BED files
    # k562
    k_emvar_bed = df2bed(k_emvars)
    # hepg2
    h_emvar_bed = df2bed(h_emvars)
    # sknsh
    s_emvar_bed = df2bed(s_emvars)
    ### third ###
    # open seqlets for each cell type
    # k562
    k_seqlets_bed = pybedtools.BedTool(path2distalCREseqlets_k562)
    # hepg2
    h_seqlets_bed = pybedtools.BedTool(path2distalCREseqlets_hepg2)
    # sknsh
    s_seqlets_bed = pybedtools.BedTool(path2distalCREseqlets_sknsh)
    ### fourth ###
    # intersect emVars with seqlets
    # k562
    k_emVar_seqlets = k_seqlets_bed.intersect(k_emvar_bed, wa=True, wb=True).to_dataframe()
    # hepg2
    h_emVar_seqlets = h_seqlets_bed.intersect(h_emvar_bed, wa=True, wb=True).to_dataframe()
    # sknsh
    s_emVar_seqlets = s_seqlets_bed.intersect(s_emvar_bed, wa=True, wb=True).to_dataframe()
    ### five ###
    # get multiTF called enhancers
    # k562
    k_multiTF = get_multiTF_enhancers_from_long_df_vClustered(create_long_form_tf_df(k_seqlets_bed.to_dataframe(), vierstra_dict))
    # hepg2
    h_multiTF = get_multiTF_enhancers_from_long_df_vClustered(create_long_form_tf_df(h_seqlets_bed.to_dataframe(), vierstra_dict))
    # sknsh
    s_multiTF = get_multiTF_enhancers_from_long_df_vClustered(create_long_form_tf_df(s_seqlets_bed.to_dataframe(), vierstra_dict))
    ### six ###
    # filter multiTF enhancers for those with variants that intersect the seqlets
    k_multiTF_emVar_seqlets = k_multiTF[k_multiTF['enhancer_id'].isin(k_emVar_seqlets['thickEnd'].unique())]
    return k_multiTF, k_emVar_seqlets, k_multiTF_emVar_seqlets

In [3]:
# open vierstra clusters for collapsing on families instead of tfs
vierstra_motifs = pd.read_excel('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/gnomad_buffering_analysis/motif_annotations.xlsx', sheet_name=[0,1])
# make a dictionary out of the clusterIDs and Names - this will be used to generate the final dictionary with the motif names
idName_dict = dict(zip(vierstra_motifs[0]['Cluster_ID'], vierstra_motifs[0]['Name']))
# open the second page and assign the Names to the individual motifs
vierstra_motifs[1].loc[:,'cluster_name'] = [idName_dict.get(i) for i in vierstra_motifs[1]['Cluster_ID']]
# make a dictionary that pulls the motif name as a key and returns the vierstra family as a value
vierstra_motif_dict = dict(zip(vierstra_motifs[1]['Motif'], vierstra_motifs[1]['cluster_name']))

In [18]:
check1, check2, check3 = multiTF_analysis('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/mpac_preds/GRCh38-dELS-chr1-ALL-mpac-017.tsv.gz', 
                 '../../../processed_data/bed_files/repTF_k562_dELS_seqlets_01.bed', 
                 '../../../processed_data/bed_files/repTF_hepg2_dELS_seqlets_01.bed', 
                 '../../../processed_data/bed_files/repTF_sknsh_dELS_seqlets_01.bed',
                 vierstra_motif_dict)

10it [00:24,  2.49s/it]


In [36]:
check3[check3['thickEnd'] == 'EH38E1311512']

,chrom,start,end,name,score,strand,thickStart,thickEnd,full_hit_string,hocomoco_tf,representative_tf,enhancer_id,rep_tf_contrib,vierstra_cluster,activity_class
3192,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,ZFX_HUMAN.H11MO.0.A,ZFX,EH38E1311512,-1.048101,ZFX,Repressor
3193,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.10387...,ZFX_HUMAN.H11MO.0.A,ZFX,EH38E1311512,-1.103878,ZFX,Repressor
3194,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,SNAI1_HUMAN.H11MO.0.C_EH38E1311512_K562_-1.201...,SNAI1_HUMAN.H11MO.0.C,SNAI1,EH38E1311512,-1.201866,Ebox/CACCTG,Repressor


In [23]:
check1[check1['enhancer_id'] == 'EH38E1311512']

,chrom,start,end,name,score,strand,thickStart,thickEnd,full_hit_string,hocomoco_tf,representative_tf,enhancer_id,rep_tf_contrib,vierstra_cluster,activity_class
3192,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,ZFX_HUMAN.H11MO.0.A,ZFX,EH38E1311512,-1.048101,ZFX,Repressor
3193,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.10387...,ZFX_HUMAN.H11MO.0.A,ZFX,EH38E1311512,-1.103878,ZFX,Repressor
3194,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,SNAI1_HUMAN.H11MO.0.C_EH38E1311512_K562_-1.201...,SNAI1_HUMAN.H11MO.0.C,SNAI1,EH38E1311512,-1.201866,Ebox/CACCTG,Repressor


In [26]:
check2[check2['thickEnd'] == 'EH38E1311512']

,chrom,start,end,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts
1650,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,chr1,2125485,2125486,chr1:2125486:G:C_EH38E1311512
1651,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,chr1,2125488,2125489,chr1:2125489:C:G_EH38E1311512
1652,chr1,2125485,2125492,ZFX_HUMAN.H11MO.0.A_EH38E1311512_K562_-1.04810...,3,SNAI1,Repressor,EH38E1311512,chr1,2125489,2125490,chr1:2125490:T:G_EH38E1311512


In [25]:
check2.head()

,chrom,start,end,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts
0,chr1,2054151,2054156,TAL1_HUMAN.H11MO.0.A_EH38E2778258_K562_2.87581...,1,TAL1,Activator,EH38E2778258,chr1,2054151,2054152,chr1:2054152:G:A_EH38E2778258
1,chr1,2054151,2054156,TAL1_HUMAN.H11MO.0.A_EH38E2778258_K562_2.87581...,1,TAL1,Activator,EH38E2778258,chr1,2054151,2054152,chr1:2054152:G:C_EH38E2778258
2,chr1,2054151,2054156,TAL1_HUMAN.H11MO.0.A_EH38E2778258_K562_2.87581...,1,TAL1,Activator,EH38E2778258,chr1,2054151,2054152,chr1:2054152:G:T_EH38E2778258
3,chr1,2054151,2054156,TAL1_HUMAN.H11MO.0.A_EH38E2778258_K562_2.87581...,1,TAL1,Activator,EH38E2778258,chr1,2054152,2054153,chr1:2054153:A:G_EH38E2778258
4,chr1,2054151,2054156,TAL1_HUMAN.H11MO.0.A_EH38E2778258_K562_2.87581...,1,TAL1,Activator,EH38E2778258,chr1,2054152,2054153,chr1:2054153:A:T_EH38E2778258


In [28]:
def read_in_sat_mut (path2satmut, chunksize=1000000):
        # define a list for concatenating #
        chunks2cat = []
        # iterate through chunked tsv #
        for chunk in tqdm(pd.read_csv(path2satmut, sep = '\t', chunksize=chunksize)):
            chunks2cat.append(chunk)
        # concatenate chunks #
        cat_df = pd.concat(chunks2cat)
        return cat_df

In [29]:
chr1_preds = read_in_sat_mut('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/mpac_preds/GRCh38-dELS-chr1-ALL-mpac-017.tsv.gz')

104it [03:49,  2.21s/it]


In [31]:
chr1_preds.head()

,chrom,pos,id,ref,alt,k562_ref_pred,k562_alt_pred,k562_skew_pred,hepg2_ref_pred,hepg2_alt_pred,hepg2_skew_pred,sknsh_ref_pred,sknsh_alt_pred,sknsh_skew_pred
0,chr1,235687951,EH38E1434469,G,A,0.580428,0.507609,-0.072820,0.410051,0.344707,-0.065344,0.499364,0.436136,-0.063229
1,chr1,235687951,EH38E1434469,G,T,0.580428,0.560662,-0.019767,0.410051,0.386293,-0.023758,0.499364,0.492938,-0.006426
2,chr1,235687951,EH38E1434469,G,C,0.580428,0.539989,-0.040439,0.410051,0.388695,-0.021356,0.499364,0.425235,-0.074129
3,chr1,235687952,EH38E1434469,T,A,0.431364,0.496984,0.065621,0.337839,0.392331,0.054493,0.461711,0.533479,0.071768
4,chr1,235687952,EH38E1434469,T,C,0.431364,0.427699,-0.003665,0.337839,0.284099,-0.053740,0.461711,0.420498,-0.041213


In [35]:
chr1_preds[(chr1_preds['id'] == 'EH38E1311512') & (chr1_preds['k562_skew_pred'].abs() > 0.5)]

,chrom,pos,id,ref,alt,k562_ref_pred,k562_alt_pred,k562_skew_pred,hepg2_ref_pred,hepg2_alt_pred,hepg2_skew_pred,sknsh_ref_pred,sknsh_alt_pred,sknsh_skew_pred
10590087,chr1,2125474,EH38E1311512,G,C,0.424016,1.432413,1.008398,0.346824,0.950311,0.603487,0.528393,1.317047,0.788655
10590108,chr1,2125481,EH38E1311512,T,G,0.487535,1.043736,0.556202,0.383348,0.953451,0.570103,0.581492,1.195781,0.614290
10590123,chr1,2125486,EH38E1311512,G,C,0.500013,1.082055,0.582042,0.428018,0.882640,0.454623,0.609054,1.371009,0.761955
10590132,chr1,2125489,EH38E1311512,C,G,0.393295,3.449338,3.056043,0.326030,1.950693,1.624662,0.539927,2.147219,1.607293
10590135,chr1,2125490,EH38E1311512,T,G,0.499031,1.334690,0.835659,0.382455,0.790147,0.407692,0.610761,1.168183,0.557422
10590145,chr1,2125494,EH38E1311512,T,A,0.461809,-0.051303,-0.513112,0.353347,0.095690,-0.257657,0.541689,0.532704,-0.008985
10590166,chr1,2125501,EH38E1311512,G,A,0.557606,1.123931,0.566325,0.417668,0.396458,-0.021210,0.570520,0.589084,0.018564
10590233,chr1,2125523,EH38E1311512,T,C,0.598424,1.469597,0.871173,0.374219,0.689850,0.315631,0.462969,0.921656,0.458687
10590246,chr1,2125527,EH38E1311512,A,G,0.559284,1.431838,0.872554,0.338122,0.796880,0.458758,0.449540,1.097307,0.647767
